# **CV融合算子开发**
本章将通过代码辅助讲解CV融合算子的基础理论，以及CV融合算子的实现：
- 环境准备
- CV融合算子分析
- CV融合算子工程创建
- CV融合算子实现
- CV融合算子测试
- 课后实践
---

## **1. 环境准备**
本文所有内容均存放于Sources文件夹。
在开始创建算子工程前，先要对jupyter环境进行初始化。以下代码完成了初始化并将环境中的变量导入jupyter环境，并完成代码目录的创建。保证能正常导入代码以及编译。

In [ ]:
!mkdir -p Sources/05.04

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

安装msopgen工具

In [ ]:
# 下载工具仓代码
!git clone https://gitcode.com/Ascend/msopgen.git
#编译安装msopgen工具
!cd msopgen; python build.py;cd output;pip install mindstudio_opgen*.whl --force-reinstall;pip install mindstudio_opst*.whl --force-reinstall


---
## **2. CV融合算子分析**
假设我们有个简单的小模型，里面包含了一个Matmul、一个Leakyrelu算子，原始的模型如下：  

<img src="./images/cv_operator_model.png">

### **2.1 执行步骤拆解**
Matmul与LeakyReLU两个算子原生采用串行执行模式，完整流程可拆解为 **6 个独立且串行的步骤**：
- 阶段1（Matmul执行）：Matmul输入数据搬入 → Cube核执行Matmul计算 → Matmul输出数据搬出  

- 阶段2（LeakyReLU执行）：LeakyReLU输入数据搬入 → Vector核执行 LeakyReLU计算 → LeakyReLU输出数据搬出

### **2.2 核心效率瓶颈**
- **计算核资源利用率低**：Matmul依赖Cube核运算，LeakyReLU依赖Vector核运算，串行执行时总有一类计算核处于完全空闲状态；  

- **数据传输开销大**：LeakyReLU输入完全依赖Matmul输出，中间结果需经历“全局内存 ↔ 计算核”的两次完整搬移，额外增加访存耗时；  

- **执行链路冗长**：6 个独立步骤串行执行，无并行优化空间，端到端耗时高。  


将Matmul与LeakyReLU融合为单一算子后，通过「数据切块+核间协同+本地缓存」重构执行流程，整体以 **5 个核心步骤循环执行** 完成全量数据运算：

<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 1485px; border: 1px solid #ddd;">
  <tr>
    <td style="width: 60px; padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">步骤</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">操作内容</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">核心目的</td>
  </tr>
  <tr>
    <td style="width: 60px; padding: 8px; border-bottom: 1px solid #ddd;">1</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">数据切块：将Matmul输入数据（a矩阵、b矩阵、bias偏置）按 Cube 核单次计算能力切分为若干数据块</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">Cube每运算完成一小块数据可以直接让Vector核开始计算，避免Vector核空闲等待</td>
  </tr>
  <tr>
    <td style="width: 60px; padding: 8px; border-bottom: 1px solid #ddd;">2</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">Cube核数据搬入：将单块输入数据直接搬入Cube核本地存储</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">数据需要放在Cube核上完成矩阵乘运算</td>
  </tr>
  <tr>
    <td style="width: 60px; padding: 8px; border-bottom: 1px solid #ddd;">3</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">Cube核并行计算Matmul：各Cube核并行完成对应数据块的Matmul计算，结果暂存于LocalTensor</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">利用Matmul API的能力，直接输出Matmul计算结果到UB，减少编写搬运代码</td>
  </tr>
  <tr>
    <td style="width: 60px; padding: 8px; border-bottom: 1px solid #ddd;">4</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">Vector核并行计算LeakyReLU：直接读取LocalTensor中的Matmul结果，完成LeakyReLU 计算并暂存</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">核间协同无数据搬移，Vector核无空闲</td>
  </tr>
  <tr>
    <td style="width: 60px; padding: 8px;">5</td>
    <td style="padding: 8px;">结果统一搬出：将LeakyReLU结果从LocalTensor搬至全局内存指定偏移地址</td>
    <td style="padding: 8px;">将计算结果放在正确的位置</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>

### **2.3 融合核心优化效果**
- **算力利用率提升**：Cube/Vector 核在同一算子调用中协同工作，大幅减少核空闲时长，提升硬件利用率；

---
## **3. CV融合算子IR创建**
融合后的算子仅包含输入a、b、bias和输出c，其计算流程为：c = leakyrelu(a * b + c, alpha)。其中a、b为half类型输入，bias为float类型输入，c为float类型输出。此处仅以M = 1024， K = 256， N = 640输入数据作为演示示例，在真实业务场景下，可灵活调整输入数据的shape参数，也可直接支持泛化输入形式。

<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 550px; border: 1px solid #ddd;">
  <!-- 表头行：算子类型 -->
  <tr>
    <td colspan="1" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">算子类型 (OpType)</td>
    <td colspan="4" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;text-align: center;" > MatmulLeakyreluCustom</td>
  </tr>
  
  <!-- 算子输入模块：纵向合并单元格 -->
  <tr>
    <!-- 算子输入 纵向合并3行（x行+y行+列名行） -->
    <td rowspan="4" style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输入</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">name</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">shape</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">data type</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">format</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">a</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(1024, 256)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">b</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(256, 640)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
   <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">bias</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(640)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">float</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr> 
  <!-- 算子输出模块：纵向合并单元格 -->

  <tr>
  <td  style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输出</td>
    <td style="padding: 8px;">c</td>
    <td style="padding: 8px;">(1024, 640)</td>
    <td style="padding: 8px;">float</td>
    <td style="padding: 8px;">ND</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>  

我们可以创建一个新的IR文件，命名为cv_fused_op.json，内容如下：

In [ ]:
%%writefile Sources/05.04/cv_fused_op.json
[
    {
        "op": "MatmulLeakyreluCustom",
        "language": "cpp",
        "input_desc": [
            {
                "name": "a",
                "param_type": "required",
                "format": ["ND"],
                "type": ["float16"]
            },
            {
                "name": "b",
                "param_type": "required",
                "format": ["ND"],
                "type": ["float16"]
            },
            {
                "name": "bias",
                "param_type": "required",
                "format": ["ND"],
                "type": ["float"]
            }
        ],
        "output_desc": [
            {
                "name": "c",
                "param_type": "required",
                "format": ["ND"],
                "type": ["float"]
            }
        ]
    }
]

创建cv_fused_op.json文件后，即可使用msopgen工具生成算子工程，具体步骤如下：

In [ ]:
# 清除已有的custom_op目录
!rm -rf Sources/05.04/custom_op

# 使用msopgen工具创建算子工程前设置使用开源仓编译的msopgen工具，不设置时默认使用CANN安装目录下的msopgen工具
!export PATH=/usr/local/bin:~/.local/bin:$PATH;msopgen gen -i Sources/05.04/cv_fused_op.json -c ai_core-ascend910b1 -lan cpp -out Sources/05.04/custom_op


命令执行完后，会创建算子工程，目录结构如下所示：
```
custom_op
|--- framework
|--- op_host
|   |--- matmul_leakyrelu_custom.cpp          // 包含算子原型注册、tiling实现 shape与Dtype推导内容
|   |--- CMakeLists.txt                       // host侧CMakeLists文件，一般不需要修改
|--- op_kernel
|   |--- matmul_leakyrelu_custom_tiling.h     // 算子tiling定义文件   
|   |--- matmul_leakyrelu_custom.cpp          // 算子代码实现文件 
|   |--- CMakeLists.txt                       // kernel侧CMakeLists文件，一般不需要修改
|--- CMakeLists.txt                           // 根目录CMakeLists文件，一般不需要修改
|--- CMakePresets.json                        // CMake编译配置文件
|--- build.sh                                 // 编译脚本
```  
工程目录中的op_kernel和op_host包含了算子的核心实现文件。op_kernel下存放kernel侧算子实现。op_host下存放host侧代码实现，包括算子原型定义、host侧tiling实现。
* **op_host/matmul_leakyrelu_custom.cpp**: 文件名会根据cv_fused_op.json内定义的算子名生成，包含算子原型注册、tiling实现 Shape与Dtype推导内容。
* **op_kernel/matmul_leakyrelu_custom_tiling.h**: 文件名会根据cv_fused_op.json内定义的算子名生成，包含算子tiling结构体定义。
* **op_kernel/matmul_leakyrelu_custom.cpp**: 文件名会根据cv_fused_op.json内定义的算子名生成，包含算子代码实现。
* **CMakePresets.json**: CMake编译配置文件，一般只需要修改ASCEND_CANN_PACKAGE_PATH为实际CANN安装路径。

执行以下命令查看刚刚生成的算子工程目录结构：

In [ ]:
!cd Sources/05.04/custom_op;find . -maxdepth 2 -print | sed -e 's;[^/]*/;|____;g;s;____|;    |;g'

---
## **4. CV融合算子host侧代码实现**
Matmul在Cube上的数据流向为：GM →L1→L0A/L0B →Cube →L0C→FixPipe→GM  
LeakRelu在Cube上的数据流向为：GM → UB → Vector → UB → GM
所以，在Host侧实现时，需要考虑每一次计算的数据小块大小不能超出L0C的大小和UB的大小。这里以A2设备展示：  
 
<img src="./images/a2_frame.png" alt="turing_test"  width="700px">  

首先定义tiling结构体，在op_kernel/matmul_leakyrelu_custom_tiling.h文件增加Leakyrelu需要的alpha和Matmul的cubeTilingData。

In [ ]:
%%writefile Sources/05.04/custom_op/op_kernel/matmul_leakyrelu_custom_tiling.h
#ifndef MATMUL_LEAKYRELU_CUSTOM_TILING_H
#define MATMUL_LEAKYRELU_CUSTOM_TILING_H

#include <cstdint>
#include "kernel_tiling/kernel_tiling.h"

struct MatmulLeakyreluCustomTilingData {
    float alpha;
	AscendC::tiling::TCubeTiling cubeTilingData;
};
#endif

接下来着手实现op_host/matmul_leakyrelu_custom.cpp文件的Tiling函数的开发。由于计算形状（shape）为固定值，可直接设定M、N、K的具体数值。结合此前课程内容，Matmul算子单次计算会生成尺寸为baseM * baseN的子矩阵（小矩阵），因此在配置baseM 和 baseN时，需满足核心约束条件：  
baseM * baseN * sizeof(float) ≤ L0 Cache（L0C）的存储容量。  

同时，LeakRelu算子的输入数据直接来源于Matmul算子的输出结果，因此其输入数据量需满足：  
baseM * baseN * sizeof(float) ≤ Unified Buffer（UB）的存储容量。  

为简化演示流程，本示例中我们指定仅使用1个AiCore完成运算。**分离模式下Matmul API均由AIV侧发起，调用Iterate时AIV仅通知AIC执行矩阵计算，计算完成后由AIC回传结束信号；SetBlockDim配置计算所用AI Core总数，SetDim配置参与计算的AIV数量，Atlas A2单AI Core内Cube核与Vector核配比为1:2，因此需将TCubeTiling的SetDim设为2、TilingContext的SetBlockDim设为1。**。

In [ ]:
%%writefile Sources/05.04/custom_op/op_host/matmul_leakyrelu_custom.cpp
#include "../op_kernel/matmul_leakyrelu_custom_tiling.h"
#include "register/op_def_registry.h"
#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"
using namespace matmul_tiling;

namespace optiling {
static ge::graphStatus TilingFunc(gert::TilingContext* context)
{
    MatmulLeakyreluCustomTilingData *tiling = context->GetTilingData<MatmulLeakyreluCustomTilingData>();
    int32_t M = 1024;
    int32_t N = 640;
    int32_t K = 256;
    int32_t baseM = 128;
    int32_t baseN = 128;
    auto ascendcPlatform = platform_ascendc::PlatformAscendC(context->GetPlatformInfo());
    MultiCoreMatmulTiling cubeTiling(ascendcPlatform);
    cubeTiling.SetDim(2); // Set the number of cores that participate in multi-core computaion is 2.
    cubeTiling.SetAType(TPosition::GM, CubeFormat::ND, DataType::DT_FLOAT16);
    cubeTiling.SetBType(TPosition::GM, CubeFormat::ND, DataType::DT_FLOAT16);
    cubeTiling.SetCType(TPosition::VECIN, CubeFormat::ND, DataType::DT_FLOAT);
    cubeTiling.SetBiasType(TPosition::GM, CubeFormat::ND, DataType::DT_FLOAT);
    cubeTiling.SetShape(M, N, K);
    cubeTiling.SetOrgShape(M, N, K);
    cubeTiling.SetFixSplit(baseM, baseN, -1); // Set the fixed baseM=128, baseN=128.
    cubeTiling.SetBias(true);
    cubeTiling.SetBufferSpace(-1, -1, -1);
    if (cubeTiling.GetTiling(tiling->cubeTilingData) == -1) { // Get matmul tiling data.
        return ge::GRAPH_FAILED;
    }
    tiling->alpha = 0.001; // Set the leakyrelu tiling alpha=0.001.
    context->SetBlockDim(1);
    size_t userWorkspaceSize = 0;
    size_t systemWorkspaceSize = static_cast<size_t>(ascendcPlatform.GetLibApiWorkSpaceSize());
    size_t *currentWorkspace = context->GetWorkspaceSizes(1);
    currentWorkspace[0] = userWorkspaceSize + systemWorkspaceSize;
    return ge::GRAPH_SUCCESS;
}
}

namespace ge {
static ge::graphStatus InferShape(gert::InferShapeContext* context)
{
    const gert::Shape* x1_shape = context->GetInputShape(0);
    gert::Shape* y_shape = context->GetOutputShape(0);
    *y_shape = *x1_shape;
    return GRAPH_SUCCESS;
}
static ge::graphStatus InferDataType(gert::InferDataTypeContext *context)
{
    const auto inputDataType = context->GetInputDataType(0);
    context->SetOutputDataType(0, inputDataType);
    return ge::GRAPH_SUCCESS;
}
}

namespace ops {
class MatmulLeakyreluCustom : public OpDef {
public:
    explicit MatmulLeakyreluCustom(const char* name) : OpDef(name)
    {
        this->Input("a")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16})
            .Format({ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND});
        this->Input("b")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16})
            .Format({ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND});
        this->Input("bias")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT})
            .Format({ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND});
        this->Output("c")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT})
            .Format({ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND});

        this->SetInferShape(ge::InferShape).SetInferDataType(ge::InferDataType);

        this->AICore()
            .SetTiling(optiling::TilingFunc);
        this->AICore().AddConfig("ascend910b");
    }
};

OP_ADD(MatmulLeakyreluCustom);
}

---
## **5. CV融合算子kernel侧代码实现**
最后在op_kernel/matmul_leakyrelu_custom.cpp完成CV融合算子的Kernel实现开发。
MatmulLeakyRelu融合算子的整体计算流程为：Matmul计算数据搬入->Matmul计算->Matmul计算结果搬出->LeakyRelu计算数据搬入->LeakyRelu计算->LeakyRelu计算结果搬出。  

<img src="./images/data_trends.png"  alt="turing_test"  width="700px">    

其中Matmul计算数据搬入、Matmul计算结果搬出和LeakyRelu计算数据搬入的操作可以通过Matmul API直接实现。 所以我们可以定义算子类包含以下函数：  
- MatmulCompute : 实现Matmul计算逻辑。
- LeakyReluCompute : 实现LeakyRelu计算逻辑。
- CopyOut : LeakyRelu计算结果搬出操作。CopyOut函数的实现逻辑与Vector算子运算结果的搬出流程不同。核心原因在于MatmulLeakyRelu算子的输出是一小块矩阵，导致其向GM的数据写入无法采用直接填充的方式。具体操作要求为：每完成一行小矩阵计算结果的填充后，需先跳过(整体矩阵单行元素数 - 小矩阵单行元素数)的地址偏移量，再执行下一行小矩阵结果的填充操作。    
    <img src="./images/result.png"  alt="turing_test"  width="450px">   
   
- CalcOffset: 计算偏移函数。    
    1. **分块方式**：左矩阵（A）按行维度分块，右矩阵（B）按列维度分块，单个核负责的计算任务为「左矩阵某行块 × 右矩阵某列块」，运算结果对应结果矩阵（C）中对应的子矩阵区域。  

    2. **核任务分配逻辑（以示例说明）**：  
        若左矩阵分为 A1、A2 两个行块，右矩阵分为 B1、B2 两个列块，共启用 4 个核运算，则按 M 方向遍历规则分配任务：先按核编号依次遍历左矩阵所有行块，完成一轮遍历后，右矩阵列块向右滑动一列进入下一轮。具体分配为：  
        Core0：A1 × B1  
        Core1：A2 × B1  
        Core2：A1 × B2  
        Core3：A2 × B2   
        <img src="./images/offset.png"  alt="turing_test"  width="400px">     

    3. **分块索引计算**：  
    当前核处理的左矩阵行块 = 核编号 % 左矩阵总块数  
    当前核处理的右矩阵列块 = 核编号 / 左矩阵总块数（整除）  
    A 偏移 = 左块索引 × 单块行数 × A 列数（转置则为左块索引 × 单块行数）  
    B 偏移 = 右块索引 × 单块列数（转置则为右块索引 × 单块列数 × B 行数）  
    C 偏移 = 左块起始行 × C 总列数 + 右块起始列  

- Init：根据CalcOffset函数计算得到当前核处理的A矩阵偏移、B矩阵偏移、C矩阵偏移。  
- Process：设置当前核处理的A矩阵、B矩阵、C矩阵指针，并通过Iterate获取每次运算得到的C矩阵小块结果，用于LeakyRelu计算。


In [ ]:
%%writefile Sources/05.04/custom_op/op_kernel/matmul_leakyrelu_custom.cpp
#include "kernel_operator.h"
#include "matmul_leakyrelu_custom_tiling.h"
#include "lib/matmul_intf.h"

using namespace matmul;

__aicore__ inline uint32_t Ceiling(uint32_t a, uint32_t b)
{
    return (a + b - 1) / b;
}

template <typename aType, typename bType, typename cType, typename biasType> class MatmulLeakyKernel {
public:
    __aicore__ inline MatmulLeakyKernel(){};
    __aicore__ inline void Init(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c, GM_ADDR workspace,
                                const TCubeTiling &tiling, float alpha, AscendC::TPipe *pipe);
    __aicore__ inline void Process(AscendC::TPipe *pipe);

    __aicore__ inline void MatmulCompute();
    __aicore__ inline void LeakyReluCompute();
    __aicore__ inline void CopyOut(uint32_t count);
    __aicore__ inline void CalcOffset(int32_t blockIdx, const TCubeTiling &tiling, int32_t &offsetA, int32_t &offsetB,
                                      int32_t &offsetC, int32_t &offsetBias);

    Matmul<MatmulType<AscendC::TPosition::GM, CubeFormat::ND, aType>, MatmulType<AscendC::TPosition::GM, CubeFormat::ND, bType>,
           MatmulType<AscendC::TPosition::VECIN, CubeFormat::ND, cType>, MatmulType<AscendC::TPosition::GM, CubeFormat::ND, biasType>>
        matmulObj;

    AscendC::GlobalTensor<aType> aGlobal;
    AscendC::GlobalTensor<bType> bGlobal;
    AscendC::GlobalTensor<cType> cGlobal;
    AscendC::GlobalTensor<biasType> biasGlobal;
    AscendC::LocalTensor<cType> reluOutLocal;
    float alpha;
    TCubeTiling tiling;
    AscendC::TQue<AscendC::QuePosition::VECOUT, 1> reluOutQueue_;
};

template <typename aType, typename bType, typename cType, typename biasType>
__aicore__ inline void
MatmulLeakyKernel<aType, bType, cType, biasType>::Init(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c, GM_ADDR workspace,
                                                       const TCubeTiling &tiling, float alpha, AscendC::TPipe *pipe)
{
    this->tiling = tiling;
    this->alpha = alpha;
    aGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ aType *>(a), tiling.M * tiling.Ka);
    bGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ bType *>(b), tiling.Kb * tiling.N);
    cGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ cType *>(c), tiling.M * tiling.N);
    biasGlobal.SetGlobalBuffer(reinterpret_cast<__gm__ biasType *>(bias), tiling.N);

    int offsetA = 0;
    int offsetB = 0;
    int offsetC = 0;
    int offsetBias = 0;
    CalcOffset(AscendC::GetBlockIdx(), tiling, offsetA, offsetB, offsetC, offsetBias); // Calculate the gm offset based on the blockidx.
    aGlobal = aGlobal[offsetA];
    bGlobal = bGlobal[offsetB];
    cGlobal = cGlobal[offsetC];
    biasGlobal = biasGlobal[offsetBias];
    pipe->InitBuffer(reluOutQueue_, 1, tiling.baseM * tiling.baseN * sizeof(cType)); // Init output buffer.
    if (GetSysWorkSpacePtr() == nullptr) {
        return;
    }
}

template <typename aType, typename bType, typename cType, typename biasType>
__aicore__ inline void MatmulLeakyKernel<aType, bType, cType, biasType>::Process(AscendC::TPipe *pipe)
{
    uint32_t computeRound = 0;
    matmulObj.SetTensorA(aGlobal);
    matmulObj.SetTensorB(bGlobal);
    matmulObj.SetBias(biasGlobal);
    while (matmulObj.template Iterate<true>()) { // Once Iterate, compute baseM * baseN, sync is set true here.
        MatmulCompute(); // Get matmul compute result.
        LeakyReluCompute(); // Compute leakyRelu.
        CopyOut(computeRound); // Copy leakyRelu out result to GM.
        computeRound++;
    }
    matmulObj.End();
}

template <typename aType, typename bType, typename cType, typename biasType>
__aicore__ inline void MatmulLeakyKernel<aType, bType, cType, biasType>::MatmulCompute()
{
    reluOutLocal = reluOutQueue_.AllocTensor<cType>();
    matmulObj.template GetTensorC<true>(reluOutLocal, false, true);
}

template <typename aType, typename bType, typename cType, typename biasType>
__aicore__ inline void MatmulLeakyKernel<aType, bType, cType, biasType>::LeakyReluCompute()
{
    LeakyRelu(reluOutLocal, reluOutLocal, (cType)alpha, tiling.baseM * tiling.baseN);
    reluOutQueue_.EnQue(reluOutLocal);
}

template <typename aType, typename bType, typename cType, typename biasType>
__aicore__ inline void MatmulLeakyKernel<aType, bType, cType, biasType>::CopyOut(uint32_t count)
{
    reluOutQueue_.DeQue<cType>();
    const uint32_t roundM = tiling.singleCoreM / tiling.baseM;
    const uint32_t roundN = tiling.singleCoreN / tiling.baseN;
    uint32_t startOffset = (count % roundM * tiling.baseM * tiling.N + count / roundM * tiling.baseN);
    DataCopyParams copyParam = {(uint16_t)tiling.baseM, (uint16_t)(tiling.baseN * sizeof(cType) / DEFAULT_C0_SIZE), 0,
                                (uint16_t)((tiling.N - tiling.baseN) * sizeof(cType) / DEFAULT_C0_SIZE)};
    DataCopy(cGlobal[startOffset], reluOutLocal, copyParam);
    reluOutQueue_.FreeTensor(reluOutLocal);
}

template <typename aType, typename bType, typename cType, typename biasType>
__aicore__ inline void
MatmulLeakyKernel<aType, bType, cType, biasType>::CalcOffset(int32_t blockIdx, const TCubeTiling &tiling,
                                                             int32_t &offsetA, int32_t &offsetB, int32_t &offsetC,
                                                             int32_t &offsetBias)
{
    auto mSingleBlocks = Ceiling(tiling.M, tiling.singleCoreM);
    auto mCoreIndx = blockIdx % mSingleBlocks;
    auto nCoreIndx = blockIdx / mSingleBlocks;
    offsetA = mCoreIndx * tiling.Ka * tiling.singleCoreM;
    offsetB = nCoreIndx * tiling.singleCoreN;
    offsetC = mCoreIndx * tiling.N * tiling.singleCoreM + nCoreIndx * tiling.singleCoreN;
    offsetBias = nCoreIndx * tiling.singleCoreN;
}

extern "C" __global__ __aicore__ void matmul_leakyrelu_custom(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c, GM_ADDR workspace, GM_ADDR tiling) {
    REGISTER_TILING_DEFAULT(MatmulLeakyreluCustomTilingData);
    GET_TILING_DATA(tilingData, tiling);
    MatmulLeakyKernel<half, half, float, float> matmulLeakyKernel;
    AscendC::TPipe pipe;
    REGIST_MATMUL_OBJ(&pipe, GetSysWorkSpacePtr(), matmulLeakyKernel.matmulObj, &tilingData.cubeTilingData); // Initialize the matmul object.
    matmulLeakyKernel.Init(a, b, bias, c, workspace, tilingData.cubeTilingData, tilingData.alpha, &pipe);
    matmulLeakyKernel.Process(&pipe);
   
}

---
## **6. CV融合算子测试**
完成CV融合算子的Host侧与Kernel侧代码后，需要先把算子编译部署。

In [ ]:
!cd Sources/05.04/custom_op;bash build.sh;./build_out/custom_opp_*.run --install-path=${HOME}/

接下来编写测试代码，这里以固定的M = 1024 ，N = 640， K = 256。 a矩阵全1，b矩阵全2数据进行测试。

In [ ]:
%%writefile Sources/05.04/aclnn_test.cpp
#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <vector>

#include "aclnn/aclnn_base.h"
#include "aclnn/acl_meta.h"
#include "acl/acl_rt.h"
#include "aclnn_matmul_leakyrelu_custom.h"

#define CHECK_ACL(expr)                                                                                 \
    do {                                                                                                \
        auto __ret = (expr);                                                                            \
        int32_t __code = static_cast<int32_t>(__ret);                                                   \
        if (__code != 0) {                                                                              \
            fprintf(stderr, "[ERROR] %s failed at %s:%d, ret=%d\n", #expr, __FILE__, __LINE__, __code); \
        }                                                                                               \
    } while (0)

int32_t main(int32_t argc, char** argv)
{
    const int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    CHECK_ACL(aclnnInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    CHECK_ACL(aclrtCreateStream(&stream));

    // 定义MatmulLeakyrelu的张量形状
    const std::vector<int64_t> shape_a = {1024, 256};    // 输入a形状
    const std::vector<int64_t> shape_b = {256, 640};     // 输入b形状
    const std::vector<int64_t> shape_bias = {640};       // 输入bias形状
    const std::vector<int64_t> shape_output = {1024, 640};// 输出形状

    // 计算各张量元素数量和内存大小
    const int64_t elementCount_a = shape_a[0] * shape_a[1];
    const int64_t elementCount_b = shape_b[0] * shape_b[1];
    const int64_t elementCount_bias = shape_bias[0];
    const int64_t elementCount_output = shape_output[0] * shape_output[1];
    
    const size_t bufferSize_a = elementCount_a * sizeof(aclFloat16);
    const size_t bufferSize_b = elementCount_b * sizeof(aclFloat16);
    const size_t bufferSize_bias = elementCount_bias * sizeof(float);
    const size_t bufferSize_output = elementCount_output * sizeof(float);

    // 分配输入a的设备内存并创建张量
    void* inputADeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&inputADeviceMem, bufferSize_a, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* inputA = aclCreateTensor(shape_a.data(), shape_a.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                        shape_a.data(), shape_a.size(), inputADeviceMem);

    // 分配输入b的设备内存并创建张量
    void* inputBDeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&inputBDeviceMem, bufferSize_b, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* inputB = aclCreateTensor(shape_b.data(), shape_b.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                        shape_b.data(), shape_b.size(), inputBDeviceMem);

    // 分配输入bias的设备内存并创建张量
    void* inputBiasDeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&inputBiasDeviceMem, bufferSize_bias, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* inputBias = aclCreateTensor(shape_bias.data(), shape_bias.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                           shape_bias.data(), shape_bias.size(), inputBiasDeviceMem);

    // 分配输出的设备内存并创建张量
    void* outputDeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&outputDeviceMem, bufferSize_output, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* output = aclCreateTensor(shape_output.data(), shape_output.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                        shape_output.data(), shape_output.size(), outputDeviceMem);

    // 准备主机测试数据（沿用原始代码的aclFloatToFloat16函数）
    std::vector<aclFloat16> inputAHostData(elementCount_a, aclFloatToFloat16(1.0));
    std::vector<aclFloat16> inputBHostData(elementCount_b, aclFloatToFloat16(2.0));
    std::vector<float> inputBiasHostData(elementCount_bias, float(0.5));
    std::vector<float> outputHostData(elementCount_output, float(0.0));
    
    // 计算预期结果：LeakyReLU(1*2*256 + 0.5, 0.001) = 512.5
    std::vector<float> goldenData(elementCount_output, float(512.5));

    // 主机数据拷贝到设备
    CHECK_ACL(aclrtMemcpy(inputADeviceMem, bufferSize_a, inputAHostData.data(),
                          bufferSize_a, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBDeviceMem, bufferSize_b, inputBHostData.data(),
                          bufferSize_b, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBiasDeviceMem, bufferSize_bias, inputBiasHostData.data(),
                          bufferSize_bias, ACL_MEMCPY_HOST_TO_DEVICE));

    // 获取MatmulLeakyrelu算子工作空间大小和执行器
    uint64_t workspaceSize = 0;
    aclOpExecutor* executor = nullptr;
    CHECK_ACL(aclnnMatmulLeakyreluCustomGetWorkspaceSize(inputA, inputB, inputBias, output, &workspaceSize, &executor));

    // 分配工作空间内存
    void* workspaceDeviceMem = nullptr;
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtMalloc(&workspaceDeviceMem, workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST));
    }

    // 执行MatmulLeakyrelu算子
    CHECK_ACL(aclnnMatmulLeakyreluCustom(workspaceDeviceMem, workspaceSize, executor, stream));
    CHECK_ACL(aclrtSynchronizeStream(stream));

    // 设备结果拷贝回主机
    CHECK_ACL(aclrtMemcpy(outputHostData.data(), bufferSize_output, outputDeviceMem,
                          bufferSize_output, ACL_MEMCPY_DEVICE_TO_HOST));

    // 打印结果并验证
    printf("result is:\n");
    const int64_t previewCount = std::min<int64_t>(elementCount_output, 10);
    for (int64_t i = 0; i < previewCount; i++) { 
        printf("%.1f ", outputHostData[i]); 
    }
    printf("\ntest %s\n", std::equal(outputHostData.begin(), outputHostData.end(), goldenData.begin()) ? "pass" : "failed");

    // 释放资源
    aclDestroyTensor(inputA);
    aclDestroyTensor(inputB);
    aclDestroyTensor(inputBias);
    aclDestroyTensor(output);
    
    CHECK_ACL(aclrtFree(inputADeviceMem));
    CHECK_ACL(aclrtFree(inputBDeviceMem));
    CHECK_ACL(aclrtFree(inputBiasDeviceMem));
    CHECK_ACL(aclrtFree(outputDeviceMem));
    
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtFree(workspaceDeviceMem));
    }
    
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclnnFinalize());
    
    return 0;
}

编译并运行测试代码，检查输出是否与预期一致。

In [ ]:
# 部署算子后编译调用代码
!g++ -I$ASCEND_TOOLKIT_HOME/include -I${HOME}/vendors/customize/op_api/include -L$ASCEND_TOOLKIT_HOME/lib64 -L${HOME}/vendors/customize/op_api/lib Sources/05.04/aclnn_test.cpp -lcust_opapi -lnnopbase -lacl_rt -o Sources/05.04/execute_op;
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;./Sources/05.04/execute_op

调用成功后，会有如下输出：
```
result is:
512.5 512.5 512.5 512.5 512.5 512.5 512.5 512.5 512.5 512.5 
test pass
```

---
# **课后实践**
尝试修改代码，开发一个新的CV融合算子，实现Matmul + Abs的计算。公式为：
<div style="width: 10%; float: left; clear: both; margin: 10px 0;">

$$
C = \text{abs}(A \times B)
$$
</div>
<div style="clear: both;"></div>

支持M = 1024, N = 640, K = 256的half类型输入，float类型输出，算子原型为：

<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 550px; border: 1px solid #ddd;">
  <!-- 表头行：算子类型 -->
  <tr>
    <td colspan="1" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">算子类型 (OpType)</td>
    <td colspan="4" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;text-align: center;" > MatmulAbs</td>
  </tr>
  
  <!-- 算子输入模块：纵向合并单元格 -->
  <tr>
    <!-- 算子输入 纵向合并3行（x行+y行+列名行） -->
    <td rowspan="4" style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输入</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">name</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">shape</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">data type</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">format</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">a</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(1024, 256)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">b</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(256, 640)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
   <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">bias</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(640)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">float</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr> 
  <!-- 算子输出模块：纵向合并单元格 -->

  <tr>
  <td  style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输出</td>
    <td style="padding: 8px;">c</td>
    <td style="padding: 8px;">(1024, 640)</td>
    <td style="padding: 8px;">float</td>
    <td style="padding: 8px;">ND</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>  

### **算子原型json文件**
原型文件不需要修改。

In [ ]:
%%writefile Sources/05.04/matmul_abs.json
[
    {
        "op": "MatmulAbs",
        "language": "cpp",
        "input_desc": [
            {
                "name": "a",
                "param_type": "required",
                "format": ["ND"],
                "type": ["float16"]
            },
            {
                "name": "b",
                "param_type": "required",
                "format": ["ND"],
                "type": ["float16"]
            },
            {
                "name": "bias",
                "param_type": "required",
                "format": ["ND"],
                "type": ["float"]
            }
        ],
        "output_desc": [
            {
                "name": "c",
                "param_type": "required",
                "format": ["ND"],
                "type": ["float"]
            }
        ]
    }
]

### **创建算子工程**
根据上文的算子原型json文件，创建算子工程。仅需执行以下命令，不需要修改。

In [ ]:
# 清除已有的custom_op目录
!rm -rf Sources/05.04/custom_op

# 使用msopgen工具创建算子工程前设置使用开源仓编译的msopgen工具，不设置时默认使用CANN安装目录下的msopgen工具
!export PATH=/usr/local/bin:~/.local/bin:$PATH;msopgen gen -i Sources/05.04/matmul_abs.json -c ai_core-ascend910b1 -lan cpp -out Sources/05.04/custom_op


### **修改Tiling结构体定义**
参照课程中MatmulLeakyreluCustom的Tiling结构体定义，修改MatmulAbsCustom的Tiling结构体定义。

In [ ]:
%%writefile Sources/05.04/custom_op/op_kernel/matmul_abs_tiling.h
#ifndef MATMUL_ABS_TILING_H
#define MATMUL_ABS_TILING_H
#include <cstdint>
#include "kernel_tiling/kernel_tiling.h"

struct MatmulAbsTilingData {
    uint32_t size;
};
#endif

### **修改host侧实现**
参照课程中MatmulLeakyreluCustom的host侧实现，修改MatmulAbsCustom的host侧实现。

In [ ]:
%%writefile Sources/05.04/custom_op/op_host/matmul_abs.cpp
#include "../op_kernel/matmul_abs_tiling.h"
#include "register/op_def_registry.h"
#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"
using namespace matmul_tiling;

namespace optiling {
static ge::graphStatus TilingFunc(gert::TilingContext* context)
{

  MatmulAbsTilingData *tiling = context->GetTilingData<MatmulAbsTilingData>();
  const gert::StorageShape* x1_shape = context->GetInputShape(0);
  int32_t data_sz = 1;
  for (int i = 0; i < x1_shape->GetStorageShape().GetDimNum(); i++)
    data_sz *= x1_shape->GetStorageShape().GetDim(i);
  tiling->size = data_sz;
  context->SetBlockDim(8);
  size_t *currentWorkspace = context->GetWorkspaceSizes(1);
  currentWorkspace[0] = 0;
  return ge::GRAPH_SUCCESS;
}
}

namespace ge {
static ge::graphStatus InferShape(gert::InferShapeContext* context)
{
    const gert::Shape* x1_shape = context->GetInputShape(0);
    gert::Shape* y_shape = context->GetOutputShape(0);
    *y_shape = *x1_shape;
    return GRAPH_SUCCESS;
}
static ge::graphStatus InferDataType(gert::InferDataTypeContext *context)
{
    const auto inputDataType = context->GetInputDataType(0);
    context->SetOutputDataType(0, inputDataType);
    return ge::GRAPH_SUCCESS;
}
}

namespace ops {
class MatmulAbs : public OpDef {
public:
    explicit MatmulAbs(const char* name) : OpDef(name)
    {
        this->Input("a")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16})
            .Format({ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND});
        this->Input("b")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT16})
            .Format({ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND});
        this->Input("bias")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT})
            .Format({ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND});
        this->Output("c")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT})
            .Format({ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND});
        this->SetInferShape(ge::InferShape).SetInferDataType(ge::InferDataType);
        this->AICore()
            .SetTiling(optiling::TilingFunc);
        this->AICore().AddConfig("ascend910b");
    }
};
OP_ADD(MatmulAbs);
}

### **Kernel侧实现**
参照课程中MatmulLeakyreluCustom的kernel侧实现，修改MatmulAbsCustom的kernel侧实现。

In [ ]:
%%writefile Sources/05.04/custom_op/op_kernel/matmul_abs.cpp
#include "kernel_operator.h"
#include "matmul_abs_tiling.h"
#include "lib/matmul_intf.h"

using namespace matmul;

extern "C" __global__ __aicore__ void matmul_abs(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c, GM_ADDR workspace, GM_ADDR tiling) {
    REGISTER_TILING_DEFAULT(MatmulAbsTilingData);
    GET_TILING_DATA(tilingData, tiling);

}

### **测试**
以下代码为MatmulAbs算子的测试代码，代码进行m = 1024, n = 640, k = 256的测试。完成该测试后可以尝试修改算子支持其他Shape，并修改该测试代码进行测试。

In [ ]:
%%writefile Sources/05.04/aclnn_test.cpp
#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <vector>

#include "aclnn/aclnn_base.h"
#include "aclnn/acl_meta.h"
#include "acl/acl_rt.h"
#include "aclnn_matmul_abs.h"

#define CHECK_ACL(expr)                                                                                 \
    do {                                                                                                \
        auto __ret = (expr);                                                                            \
        int32_t __code = static_cast<int32_t>(__ret);                                                   \
        if (__code != 0) {                                                                              \
            fprintf(stderr, "[ERROR] %s failed at %s:%d, ret=%d\n", #expr, __FILE__, __LINE__, __code); \
        }                                                                                               \
    } while (0)

int32_t main(int32_t argc, char** argv)
{
    const int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    CHECK_ACL(aclnnInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    CHECK_ACL(aclrtCreateStream(&stream));

    // 定义MatmulAbs的张量形状
    const std::vector<int64_t> shape_a = {1024, 256};    // 输入a形状
    const std::vector<int64_t> shape_b = {256, 640};     // 输入b形状
    const std::vector<int64_t> shape_bias = {640};       // 输入bias形状
    const std::vector<int64_t> shape_output = {1024, 640};// 输出形状

    // 计算各张量元素数量和内存大小
    const int64_t elementCount_a = shape_a[0] * shape_a[1];
    const int64_t elementCount_b = shape_b[0] * shape_b[1];
    const int64_t elementCount_bias = shape_bias[0];
    const int64_t elementCount_output = shape_output[0] * shape_output[1];
    
    const size_t bufferSize_a = elementCount_a * sizeof(aclFloat16);
    const size_t bufferSize_b = elementCount_b * sizeof(aclFloat16);
    const size_t bufferSize_bias = elementCount_bias * sizeof(float);
    const size_t bufferSize_output = elementCount_output * sizeof(float);

    // 分配输入a的设备内存并创建张量
    void* inputADeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&inputADeviceMem, bufferSize_a, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* inputA = aclCreateTensor(shape_a.data(), shape_a.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                        shape_a.data(), shape_a.size(), inputADeviceMem);

    // 分配输入b的设备内存并创建张量
    void* inputBDeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&inputBDeviceMem, bufferSize_b, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* inputB = aclCreateTensor(shape_b.data(), shape_b.size(), ACL_FLOAT16, nullptr, 0, ACL_FORMAT_ND,
                                        shape_b.data(), shape_b.size(), inputBDeviceMem);

    // 分配输入bias的设备内存并创建张量
    void* inputBiasDeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&inputBiasDeviceMem, bufferSize_bias, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* inputBias = aclCreateTensor(shape_bias.data(), shape_bias.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                           shape_bias.data(), shape_bias.size(), inputBiasDeviceMem);

    // 分配输出的设备内存并创建张量
    void* outputDeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&outputDeviceMem, bufferSize_output, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* output = aclCreateTensor(shape_output.data(), shape_output.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                        shape_output.data(), shape_output.size(), outputDeviceMem);

    // 准备主机测试数据（沿用原始代码的aclFloatToFloat16函数）
    std::vector<aclFloat16> inputAHostData(elementCount_a, aclFloatToFloat16(-1.0));
    std::vector<aclFloat16> inputBHostData(elementCount_b, aclFloatToFloat16(2.0));
    std::vector<float> inputBiasHostData(elementCount_bias, float(0.5));
    std::vector<float> outputHostData(elementCount_output, float(0.0));
    
    // 计算预期结果：Abs(1*2*256 + 0.5, 0.001) = 511.5
    std::vector<float> goldenData(elementCount_output, float(511.5));

    // 主机数据拷贝到设备
    CHECK_ACL(aclrtMemcpy(inputADeviceMem, bufferSize_a, inputAHostData.data(),
                          bufferSize_a, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBDeviceMem, bufferSize_b, inputBHostData.data(),
                          bufferSize_b, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBiasDeviceMem, bufferSize_bias, inputBiasHostData.data(),
                          bufferSize_bias, ACL_MEMCPY_HOST_TO_DEVICE));

    // 获取MatmulAbs算子工作空间大小和执行器
    uint64_t workspaceSize = 0;
    aclOpExecutor* executor = nullptr;
    CHECK_ACL(aclnnMatmulAbsGetWorkspaceSize(inputA, inputB, inputBias, output, &workspaceSize, &executor));

    // 分配工作空间内存
    void* workspaceDeviceMem = nullptr;
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtMalloc(&workspaceDeviceMem, workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST));
    }

    // 执行MatmulAbs算子
    CHECK_ACL(aclnnMatmulAbs(workspaceDeviceMem, workspaceSize, executor, stream));
    CHECK_ACL(aclrtSynchronizeStream(stream));

    // 设备结果拷贝回主机
    CHECK_ACL(aclrtMemcpy(outputHostData.data(), bufferSize_output, outputDeviceMem,
                          bufferSize_output, ACL_MEMCPY_DEVICE_TO_HOST));

    // 打印结果并验证
    printf("result is:\n");
    const int64_t previewCount = std::min<int64_t>(elementCount_output, 10);
    for (int64_t i = 0; i < previewCount; i++) { 
        printf("%.1f ", outputHostData[i]); 
    }
    printf("\ntest %s\n", std::equal(outputHostData.begin(), outputHostData.end(), goldenData.begin()) ? "pass" : "failed");

    // 释放资源
    aclDestroyTensor(inputA);
    aclDestroyTensor(inputB);
    aclDestroyTensor(inputBias);
    aclDestroyTensor(output);
    
    CHECK_ACL(aclrtFree(inputADeviceMem));
    CHECK_ACL(aclrtFree(inputBDeviceMem));
    CHECK_ACL(aclrtFree(inputBiasDeviceMem));
    CHECK_ACL(aclrtFree(outputDeviceMem));
    
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtFree(workspaceDeviceMem));
    }
    
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclnnFinalize());
    
    return 0;
}

修改完成后，重新编译部署算子，运行测试代码，检查输出是否与预期一致。

In [ ]:
# 部署算子
!cd Sources/05.04/custom_op;bash build.sh;./build_out/custom_opp_*.run --install-path=${HOME}/

# 部署算子后编译调用代码
!g++ -I$ASCEND_TOOLKIT_HOME/include -I${HOME}/vendors/customize/op_api/include -L$ASCEND_TOOLKIT_HOME/lib64 -L${HOME}/vendors/customize/op_api/lib Sources/05.04/aclnn_test.cpp -lcust_opapi -lnnopbase -lacl_rt -o Sources/05.04/execute_op;
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;./Sources/05.04/execute_op

预期输出：
```

result is:
511.5 511.5 511.5 511.5 511.5 511.5 511.5 511.5 511.5 511.5 
test pass
```

**执行以下命令查看答案**  

host侧实现

In [ ]:
!cat ./answer/05.04_answer/op_host/matmul_abs.cpp

Tiling结构体定义

In [ ]:
!cat ./answer/05.04_answer/op_kernel/matmul_abs_tiling.h

kernel实现

In [ ]:
!cat ./answer/05.04_answer/op_kernel/matmul_abs.cpp